# 药品说明书智能识别与语音播报系统

## 项目说明

针对药品说明书字体太小、老年人看不清读不懂的问题，本项目通过以下三个步骤，将药品说明书中的重点内容识别提取并语音播报：

1. **OCR 识别**：使用 PaddleOCR-VL-1.5 模型对药品说明书图片进行文字识别
2. **大模型整理**：使用 ERNIE-4.5 大模型对识别的文字进行整理，提取关键信息
3. **语音合成播报**：使用 PaddleSpeech 语音合成模型将整理后的文字转为音频文件

### 提取的关键信息包括：
1. 药品名称
2. 药品适应症
3. 药品的用法与用量
4. 药品的禁忌
5. 药品的不良反应

### 技术栈：
- OCR: PaddleOCR-VL-1.5
- LLM: ERNIE-4.5-0.3B-Paddle
- TTS: PaddleSpeech bert-base-chinese

### 内存优化（子进程模式）：
为确保内存完全释放，本系统采用**子进程模式**运行每个模型：
- 每个模型在独立的子进程中加载和执行
- 子进程完成后自动销毁，确保内存完全释放
- 主进程仅负责数据传递和流程控制，不加载模型
- 例如：OCR 在子进程运行，完成后子进程销毁，再启动 LLM 子进程

#### 目录：
- [模型下载与检查](#模型下载与检查)
- [生成参数设置](#生成参数设置)
- [OCR 模块](#OCR-模块)
- [LLM 模块](#LLM-模块)
- [TTS 模块](#TTS-模块)
- [管线编排与模型管理](#管线编排与模型管理)
- [主流程](#主流程)
- [Gradio 交互界面](#Gradio-交互界面)

%pip install -r requirements.txt

In [1]:
%pip uninstall opencc-python-reimplemented -y
%pip install opencc-python-reimplemented==0.1.6
%pip uninstall aistudio-sdk -y
%pip install aistudio-sdk==0.3.8
# PaddleSpeech use 0.2.6 with should be patched

Found existing installation: opencc-python-reimplemented 0.1.6
Uninstalling opencc-python-reimplemented-0.1.6:
  Successfully uninstalled opencc-python-reimplemented-0.1.6
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
  Using cached opencc_python_reimplemented-0.1.6-py2.py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.
Found existing installation: aistudio-sdk 0.3.8
Uninstalling aistudio-sdk-0.3.8:
  Successfully uninstalled aistudio-sdk-0.3.8
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.baidubce.com/pypi/simple/
  Using cached http://mirrors.baidubce.com/pypi/packages/cb/77/cd71a481bb7a76b0e9d0b6bf47711c627b1dd079001ea246893f19a9d04c/aistudio_sdk-0.3.8-py3-none-any.whl (62 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to resta

In [2]:
"""Patch script to fix aistudio_sdk import in paddlenlp.

Uses importlib.util.find_spec to locate paddlenlp WITHOUT importing it,
so this can be run before paddlenlp is imported to prevent the ImportError.
"""

import importlib.util
import os
import subprocess


def _find_paddlenlp_dir():
    # Method 1: find_spec (no import, just metadata)
    spec = importlib.util.find_spec("paddlenlp")
    if spec and spec.origin:
        return os.path.dirname(spec.origin)

    # Method 2: pip show as fallback
    result = subprocess.run(
        ["pip", "show", "paddlenlp"],
        capture_output=True, text=True,
    )
    for line in result.stdout.splitlines():
        if line.startswith("Location:"):
            return os.path.join(line.split(":", 1)[1].strip(), "paddlenlp")

    raise RuntimeError("Cannot locate paddlenlp installation directory")


def patch_aistudio_utils():
    pkg_dir = _find_paddlenlp_dir()
    target_file = os.path.join(pkg_dir, "transformers", "aistudio_utils.py")

    if not os.path.isfile(target_file):
        raise FileNotFoundError(f"Target file not found: {target_file}")

    old_line = "from aistudio_sdk.hub import download"
    new_line = "from aistudio_sdk import snapshot_download as download"

    with open(target_file, "r", encoding="utf-8") as f:
        content = f.read()

    if old_line not in content:
        if new_line in content:
            print("File already patched.")
        else:
            print(f"Target import not found in {target_file}")
        return

    patched = content.replace(old_line, new_line)

    with open(target_file, "w", encoding="utf-8") as f:
        f.write(patched)

    print(f"Patched: {target_file}")
    print(f"  {old_line}  =>  {new_line}")


patch_aistudio_utils()


File already patched.


## 模型下载与检查
[返回目录 ⬆️](#目录：)

从 AIStudio 下载三个模型（如果已存在则跳过），并检查模型文件是否完整。

> **注意**：此步骤仅下载和检查模型，**不加载模型到内存**。模型将在管线运行时按需加载。

In [1]:
from pathlib import Path
import subprocess

# --- OCR 模型 ---
ocr_model_dir = Path("baidu/PaddleOCR-VL-1.5")

if not ocr_model_dir.exists():
    subprocess.run(["aistudio", "download", "--model", "PaddlePaddle/PaddleOCR-VL-1.5", "--local_dir", str(ocr_model_dir)], check=True)
    print(f"OCR 模型已下载到: {ocr_model_dir}")
else:
    print(f"OCR 模型已存在: {ocr_model_dir}，跳过下载")

# --- LLM 模型 ---
llm_model_dir = Path("baidu/ERNIE-4.5-0.3B-Paddle")

if not llm_model_dir.exists():
    subprocess.run(["aistudio", "download", "--model", "PaddlePaddle/ERNIE-4.5-0.3B-Paddle", "--local_dir", str(llm_model_dir)], check=True)
    print(f"LLM 模型已下载到: {llm_model_dir}")
else:
    print(f"LLM 模型已存在: {llm_model_dir}，跳过下载")

# --- TTS 模型 ---
# PaddleSpeech bert-base-chinese 会在首次使用时自动下载
print("TTS 模型将在首次使用时自动下载")

OCR 模型已存在: baidu/PaddleOCR-VL-1.5，跳过下载
LLM 模型已存在: baidu/ERNIE-4.5-0.3B-Paddle，跳过下载
TTS 模型将在首次使用时自动下载


## 生成参数设置
[返回目录 ⬆️](#目录：)

设置模型的 `max_new_tokens` 参数，控制每个模型生成的最大 token 数量：
- **OCR max_new_tokens**：PaddleOCR-VL 识别文字时的最大生成长度，说明书内容多时建议调大
- **LLM max_new_tokens**：ERNIE 提取信息时的最大生成长度，需要更详细整理时可调大

In [2]:
# OCR 最大生成 token 数（说明书内容多时建议调大，默认 5120）
ocr_max_new_tokens = 200

# LLM 最大生成 token 数（需要更详细整理时可调大，默认 1024）
llm_max_new_tokens = 200

print(f"OCR max_new_tokens: {ocr_max_new_tokens}")
print(f"LLM max_new_tokens: {llm_max_new_tokens}")

OCR max_new_tokens: 200
LLM max_new_tokens: 200


## OCR 模块
[返回目录 ⬆️](#目录：)

包含图片分割、OCR 子进程工作函数，以及可独立执行的 `ocr_step`。

**子进程模式**：OCR 模型在独立子进程中加载和执行，完成后子进程自动销毁，确保内存完全释放。

In [3]:
import base64
import gc
import io
import logging
import math
import time
import multiprocessing as mp
from multiprocessing import Process, Queue

from PIL import Image

logger = logging.getLogger("drug_ocr")


# ---- 图片分割 ----

def split_image(image, num_splits=4, overlap_ratio=0.1):
    """Split an image into num_splits parts (NxN grid) with overlap."""
    grid_size = int(math.sqrt(num_splits))
    if grid_size * grid_size != num_splits:
        raise ValueError(f"num_splits must be a perfect square (e.g. 4, 9, 16), got: {num_splits}")

    w, h = image.size
    cell_w = w / grid_size
    cell_h = h / grid_size
    overlap_w = cell_w * overlap_ratio
    overlap_h = cell_h * overlap_ratio

    sub_images = []
    for row in range(grid_size):
        for col in range(grid_size):
            left = max(0, col * cell_w - overlap_w)
            upper = max(0, row * cell_h - overlap_h)
            right = min(w, (col + 1) * cell_w + overlap_w)
            lower = min(h, (row + 1) * cell_h + overlap_h)
            sub_img = image.crop((int(left), int(upper), int(right), int(lower)))
            sub_images.append(sub_img)

    return sub_images


# ---- OCR 子进程工作函数 ----

def ocr_worker_process(ocr_model_dir, image_data_list, max_new_tokens, result_queue):
    """Worker function for OCR subprocess - loads model, performs OCR, returns result."""
    try:
        import time
        import base64
        import io
        from PIL import Image
        from fastdeploy import LLM, SamplingParams

        # Load OCR model
        print("[OCR Worker] 加载 OCR 模型 (PaddleOCR-VL)...")
        start = time.perf_counter()
        ocr_model = LLM(
            model=ocr_model_dir,
            tensor_parallel_size=1,
            max_model_len=8192,
            block_size=16,
            quantization="wint8",
            graph_optimization_config={"use_cudagraph": False},
        )
        elapsed = time.perf_counter() - start
        print(f"[OCR Worker] OCR 模型加载完成, 耗时: {elapsed:.2f}s")

        # Process each image
        all_ocr_texts = []
        for i, img_bytes in enumerate(image_data_list):
            image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            print(f"[OCR Worker] 识别图片 {i+1}/{len(image_data_list)}, 尺寸: {image.size}")

            # Prepare image for OCR
            buf = io.BytesIO()
            image.save(buf, format="PNG")
            base64_image = base64.b64encode(buf.getvalue()).decode("utf-8")
            image_url = f"data:image/png;base64,{base64_image}"

            prompts = [{
                "messages": [{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": image_url}},
                        {"type": "text", "text": "OCR:"},
                    ],
                }]
            }]
            sampling_params = SamplingParams(
                temperature=0.8, top_p=0.95, max_tokens=max_new_tokens,
            )
            outputs = ocr_model.generate(prompts, sampling_params)
            response = outputs[0].outputs.text
            all_ocr_texts.append(response)
            print(f"[OCR Worker] 图片 {i+1} 识别完成, 文字长度: {len(response)}")

        # Combine results
        combined_text = "\n\n".join(all_ocr_texts)
        print(f"[OCR Worker] 全部识别完成, 总文字长度: {len(combined_text)}")

        # Put result in queue
        result_queue.put(("success", combined_text))

        # Clean up
        del ocr_model
        import gc
        gc.collect()
        print("[OCR Worker] OCR 模型已释放")

    except Exception as e:
        import traceback
        result_queue.put(("error", str(e) + "\n" + traceback.format_exc()))


# ---- 独立 OCR 步骤 (使用子进程) ----

def ocr_step(
    ocr_model_dir,
    image_path,
    enable_split=True,
    num_splits=4,
    overlap_ratio=0.1,
    max_new_tokens=5120,
):
    """Execute the OCR step in a subprocess: load image, optionally split, and run OCR."""
    step_start = time.perf_counter()
    logger.info("[OCR Step] 加载图片...")
    image = Image.open(image_path).convert("RGB")
    logger.info("[OCR Step] 图片加载完成, 尺寸: %s", image.size)

    if enable_split:
        logger.info("[OCR Step] 图片分割 (num_splits=%d, overlap=%.2f)...", num_splits, overlap_ratio)
        sub_images = split_image(image, num_splits=num_splits, overlap_ratio=overlap_ratio)
        ocr_images = [image] + sub_images
        logger.info("[OCR Step] 图片分割完成, 原始1张 + 分割%d张 = 共%d张", len(sub_images), len(ocr_images))
    else:
        logger.info("[OCR Step] 跳过图片分割")
        ocr_images = [image]

    # Serialize images to bytes for subprocess
    image_data_list = []
    for img in ocr_images:
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        image_data_list.append(buf.getvalue())

    # Create subprocess for OCR
    logger.info("[OCR Step] 启动 OCR 子进程...")
    result_queue = Queue()
    ocr_process = Process(
        target=ocr_worker_process,
        args=(str(ocr_model_dir), image_data_list, max_new_tokens, result_queue)
    )
    ocr_process.start()

    # Wait for result
    status, result = result_queue.get()
    ocr_process.join()
    ocr_process.close()

    if status == "error":
        logger.error("[OCR Step] OCR 子进程执行失败: %s", result)
        raise RuntimeError(f"OCR subprocess failed: {result}")

    combined_ocr_text = result
    logger.info("[OCR Step] OCR 识别全部完成, 总文字长度: %d, 耗时: %.2fs", len(combined_ocr_text), time.perf_counter() - step_start)

    return {"ocr_text": combined_ocr_text, "ocr_images": ocr_images}

print("✅ OCR 模块定义完成 (子进程模式)")

✅ OCR 模块定义完成 (子进程模式)


## LLM 模块
[返回目录 ⬆️](#目录：)

包含文本清洗（`clean_for_tts`）、LLM 子进程工作函数，以及可独立执行的 `llm_step`。

**子进程模式**：LLM 模型在独立子进程中加载和执行，完成后子进程自动销毁，确保内存完全释放。

In [4]:
import re
from multiprocessing import Process, Queue


# ---- 文本清洗 ----

def clean_for_tts(text):
    """Clean text for TTS synthesis by removing emojis and markdown formatting."""
    # Remove emojis (Unicode ranges for common emojis)
    # NOTE: Must avoid ranges that overlap with CJK characters (U+4E00-U+9FFF)
    text = re.sub(
        r"[\U0001F600-\U0001F64F"  # emoticons
        r"\U0001F300-\U0001F5FF"   # symbols & pictographs
        r"\U0001F680-\U0001F6FF"   # transport & map
        r"\U0001F1E0-\U0001F1FF"   # flags
        r"\U00002702-\U000027B0"   # dingbats
        r"\U000024C2-\U0000324F"   # enclosed alphanumerics (stop before CJK)
        r"\U0001F200-\U0001F251"   # enclosed CJK supplement (above CJK range)
        r"\U0001F900-\U0001F9FF"   # supplemental symbols
        r"\U0001FA00-\U0001FA6F"   # chess symbols
        r"\U0001FA70-\U0001FAFF"   # symbols extended-A
        r"\U00002600-\U000026FF"   # misc symbols
        r"\U0000FE00-\U0000FE0F"   # variation selectors
        r"\U0000200D"              # zero-width joiner
        r"]+",
        "",
        text,
    )
    # Remove markdown code blocks (```...```)
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    # Remove inline code (`...`) -> content
    text = re.sub(r"`([^`\n]+)`", r"\1", text)
    # Remove markdown headers (# ## ### etc.) at line start
    text = re.sub(r"^#{1,6}\s+", "", text, flags=re.MULTILINE)
    # Remove markdown bold (**text**) -> text
    text = re.sub(r"\*\*([^*\n]+?)\*\*", r"\1", text)
    # Remove markdown bold (__text__) -> text
    text = re.sub(r"__([^_\n]+?)__", r"\1", text)
    # Remove markdown italic (*text*) -> text
    text = re.sub(r"\*([^*\n]+?)\*", r"\1", text)
    # Remove markdown italic (_text_) -> text (only when _ is at word boundary)
    text = re.sub(r"(?<!\w)_([^_\n]+?)_(?!\w)", r"\1", text)
    # Remove markdown links [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Remove markdown images ![alt](url)
    text = re.sub(r"!\[[^\]]*\]\([^)]+\)", "", text)
    # Remove markdown horizontal rules (---, ***, ___)
    text = re.sub(r"^[-*_]{3,}\s*$", "", text, flags=re.MULTILINE)
    # Remove markdown bullet list markers (- , * , + ) at line start, keep content
    text = re.sub(r"^(\s*)[-*+]\s+", r"\1", text, flags=re.MULTILINE)
    # Remove markdown numbered list markers (1. 2. etc.) at line start, keep content
    text = re.sub(r"^(\s*)\d+\.\s+", r"\1", text, flags=re.MULTILINE)
    # Remove markdown table pipes
    text = re.sub(r"\|", " ", text)
    # Remove markdown table separator lines (---:---:---)
    text = re.sub(r"^[-: ]+$", "", text, flags=re.MULTILINE)
    # Collapse multiple blank lines into one
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Strip leading/trailing whitespace per line
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)
    # Remove leading/trailing whitespace overall
    text = text.strip()
    return text


# ---- LLM 子进程工作函数 ----

def llm_worker_process(llm_model_dir, ocr_text, max_new_tokens, result_queue):
    """Worker function for LLM subprocess - loads model, extracts info, returns result."""
    try:
        import time
        from fastdeploy import LLM, SamplingParams

        # Load LLM model
        print("[LLM Worker] 加载 LLM 模型 (ERNIE)...")
        start = time.perf_counter()
        llm_model = LLM(
            model=llm_model_dir,
            tensor_parallel_size=1,
            max_model_len=8192,
            block_size=16,
            quantization="wint8",
            graph_optimization_config={"use_cudagraph": False},
        )
        elapsed = time.perf_counter() - start
        print(f"[LLM Worker] LLM 模型加载完成, 耗时: {elapsed:.2f}s")

        # Prepare prompt
        prompt_text = f"""以下是药品说明书的 OCR 识别结果，供参考：

{ocr_text}

请根据以上 OCR 识别结果，提取并整理以下关键信息，用清晰易懂的语言重新表述，方便老年人阅读理解：

1. 药品名称
2. 药品适应症（这个药治什么病）
3. 药品的用法与用量（怎么吃、吃多少）
4. 药品的禁忌（什么人不能吃、什么情况不能吃）
5. 药品的不良反应（吃药后可能出现的不舒服）

要求：
- 只输出整理后的关键信息，不要重复或复述 OCR 原文
- 用简洁、通俗的语言回答，避免使用专业术语
- 不要使用表情符号、emoji
- 不要使用markdown格式符号（如#、**、-等），直接用纯文本输出
- 用自然流畅的口语化表达，方便语音播报
- 总字数控制在 {max_new_tokens} 字以内"""



        # todo
        prompt_text = "你是谁"

        prompts = [prompt_text]
        sampling_params = SamplingParams(
            temperature=0.8, top_p=0.95, max_tokens=max_new_tokens,
        )

        print(f"[LLM Worker] 正在生成回复 (max_new_tokens={max_new_tokens})...")
        gen_start = time.perf_counter()
        outputs = llm_model.generate(prompts, sampling_params)
        result = outputs[0].outputs.text
        gen_elapsed = time.perf_counter() - gen_start

        # Clean result
        result = clean_for_tts(result)
        print(f"[LLM Worker] 信息提取完成, 生成耗时: {gen_elapsed:.2f}s, 结果长度: {len(result)}")

        # todo
        print(">>>", result)

        # Put result in queue
        result_queue.put(("success", result))

        # Clean up
        del llm_model
        import gc
        gc.collect()
        print("[LLM Worker] LLM 模型已释放")

    except Exception as e:
        import traceback
        result_queue.put(("error", str(e) + "\n" + traceback.format_exc()))


# ---- 独立 LLM 步骤 (使用子进程) ----

def llm_step(
    llm_model_dir,
    ocr_text,
    max_new_tokens=1024,
):
    """Execute the LLM extraction step in a subprocess."""
    step_start = time.perf_counter()
    logger.info("[LLM Step] LLM 大模型信息提取...")

    # Create subprocess for LLM
    logger.info("[LLM Step] 启动 LLM 子进程...")
    result_queue = Queue()
    llm_process = Process(
        target=llm_worker_process,
        args=(str(llm_model_dir), ocr_text, max_new_tokens, result_queue)
    )
    llm_process.start()

    # Wait for result
    status, result = result_queue.get()
    llm_process.join()
    llm_process.close()

    if status == "error":
        logger.error("[LLM Step] LLM 子进程执行失败: %s", result)
        raise RuntimeError(f"LLM subprocess failed: {result}")

    extracted_info = result
    logger.info("[LLM Step] LLM 信息提取完成, 结果长度: %d, 耗时: %.2fs", len(extracted_info), time.perf_counter() - step_start)

    return {"extracted_info": extracted_info}

print("✅ LLM 模块定义完成 (子进程模式)")

✅ LLM 模块定义完成 (子进程模式)


## TTS 模块
[返回目录 ⬆️](#目录：)

包含 TTS 子进程工作函数，以及可独立执行的 `tts_step`。

**子进程模式**：TTS 模型在独立子进程中加载和执行，完成后子进程自动销毁，确保内存完全释放。

In [5]:
from multiprocessing import Process, Queue
from scipy.io.wavfile import read as wav_read


# ---- TTS 子进程工作函数 ----

def tts_worker_process(text, output_path, result_queue):
    """Worker function for TTS subprocess - loads model, synthesizes speech, returns result."""
    try:
        import time
        from paddlespeech.cli.tts.infer import TTSExecutor
        from scipy.io.wavfile import read as wav_read

        # Load TTS model
        print("[TTS Worker] 加载 TTS 模型 (PaddleSpeech)...")
        start = time.perf_counter()
        tts_model = TTSExecutor()
        elapsed = time.perf_counter() - start
        print(f"[TTS Worker] TTS 模型加载完成, 耗时: {elapsed:.2f}s")

        # Synthesize speech
        print(f"[TTS Worker] 语音合成开始, 输入文字长度: {len(text)}")
        tts_model(text=text, output=output_path)

        # Read audio data
        sr, wav_data = wav_read(output_path)

        if wav_data is not None:
            audio_duration = len(wav_data) / sr
            print(f"[TTS Worker] 语音合成完成, 音频时长: {audio_duration:.2f}s, 采样率: {sr} Hz")
            result_queue.put(("success", (sr, wav_data.tolist())))  # Convert to list for serialization
        else:
            print("[TTS Worker] 语音合成失败")
            result_queue.put(("error", "TTS synthesis failed"))

        # Clean up
        del tts_model
        import gc
        gc.collect()
        print("[TTS Worker] TTS 模型已释放")

    except Exception as e:
        import traceback
        result_queue.put(("error", str(e) + "\n" + traceback.format_exc()))


# ---- 独立 TTS 步骤 (使用子进程) ----

def tts_step(
    text,
    output_path="output.wav",
):
    """Execute the TTS synthesis step in a subprocess."""
    step_start = time.perf_counter()
    logger.info("[TTS Step] TTS 语音合成...")

    # Create subprocess for TTS
    logger.info("[TTS Step] 启动 TTS 子进程...")
    result_queue = Queue()
    tts_process = Process(
        target=tts_worker_process,
        args=(text, output_path, result_queue)
    )
    tts_process.start()

    # Wait for result
    status, result = result_queue.get()
    tts_process.join()
    tts_process.close()

    if status == "error":
        logger.error("[TTS Step] TTS 子进程执行失败: %s", result)
        logger.warning("[TTS Step] TTS 语音合成失败")
        return {"audio": None}

    sr, wav_data_list = result
    import numpy as np
    wav_data = np.array(wav_data_list, dtype=np.int16)  # Convert back from list

    audio_duration = len(wav_data) / sr
    logger.info("[TTS Step] TTS 语音合成完成, 音频时长: %.2fs, 耗时: %.2fs", audio_duration, time.perf_counter() - step_start)

    return {"audio": (sr, wav_data)}

print("✅ TTS 模块定义完成 (子进程模式)")

✅ TTS 模块定义完成 (子进程模式)


## 管线编排
[返回目录 ⬆️](#目录：)

`drug_ocr_pipeline` 串联 OCR → LLM → TTS 三个步骤，每个步骤在独立子进程中执行，`make_demo` 构建 Gradio 界面。

In [6]:
import tempfile

import numpy as np
import gradio as gr
from scipy.io.wavfile import write as wav_write


def drug_ocr_pipeline(
    ocr_model_dir,
    llm_model_dir,
    image_path,
    enable_split=True,
    num_splits=4,
    overlap_ratio=0.1,
    ocr_max_new_tokens=5120,
    llm_max_new_tokens=1024,
):
    """Drug instruction leaflet intelligent recognition and voice broadcast pipeline.
    
    Uses subprocess for each model to ensure proper memory cleanup.
    """
    pipeline_start = time.perf_counter()
    logger.info("=" * 60)
    logger.info("药品说明书识别管线启动 (子进程模式)")
    logger.info("  图片路径: %s", image_path)
    logger.info("  图片分割: %s (num_splits=%d, overlap=%.2f)", enable_split, num_splits, overlap_ratio)
    logger.info("=" * 60)

    result = {}

    # Step 1: OCR (runs in subprocess, automatically cleaned up)
    ocr_result = ocr_step(
        ocr_model_dir=ocr_model_dir,
        image_path=image_path,
        enable_split=enable_split,
        num_splits=num_splits,
        overlap_ratio=overlap_ratio,
        max_new_tokens=ocr_max_new_tokens,
    )
    result["ocr_text"] = ocr_result["ocr_text"]

    # Step 2: LLM extraction (runs in subprocess, automatically cleaned up)
    llm_result = llm_step(
        llm_model_dir=llm_model_dir,
        ocr_text=ocr_result["ocr_text"],
        max_new_tokens=llm_max_new_tokens,
    )
    result["extracted_info"] = llm_result["extracted_info"]

    # Step 3: TTS synthesis (runs in subprocess, automatically cleaned up)
    tts_result = tts_step(
        text=llm_result["extracted_info"],
    )
    result["audio"] = tts_result["audio"]

    pipeline_elapsed = time.perf_counter() - pipeline_start
    logger.info("=" * 60)
    logger.info("管线执行完成, 总耗时: %.2fs", pipeline_elapsed)
    logger.info("=" * 60)

    return result


def make_demo(ocr_model_dir, llm_model_dir, ocr_max_new_tokens=5120, llm_max_new_tokens=1024):
    """Create Gradio demo for Drug OCR Pipeline."""

    def gradio_pipeline(
        image_input,
        enable_split,
        num_splits,
        overlap_ratio,
        ocr_max_tokens,
        llm_max_tokens,
        progress=gr.Progress(track_tqdm=True),
    ):
        """Gradio interface main processing function"""
        if image_input is None:
            return "请上传药品说明书图片", "", None

        # Convert uploaded image to PIL Image
        if isinstance(image_input, str):
            image = Image.open(image_input).convert("RGB")
        else:
            image = Image.fromarray(image_input).convert("RGB") if not isinstance(image_input, Image.Image) else image_input

        # Save as temp file for pipeline
        with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
            image.save(tmp.name)
            tmp_path = tmp.name

        try:
            result = drug_ocr_pipeline(
                ocr_model_dir=ocr_model_dir,
                llm_model_dir=llm_model_dir,
                image_path=tmp_path,
                enable_split=enable_split,
                num_splits=int(num_splits),
                overlap_ratio=overlap_ratio,
                ocr_max_new_tokens=int(ocr_max_tokens),
                llm_max_new_tokens=int(llm_max_tokens),
            )

            ocr_text = result["ocr_text"]
            extracted_info = result["extracted_info"]

            # Save audio as temp file
            audio_path = None
            if result["audio"] is not None:
                sr, wav_data = result["audio"]
                audio_tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
                wav_write(audio_tmp.name, sr, wav_data.astype(np.float32))
                audio_path = audio_tmp.name

            return ocr_text, extracted_info, audio_path
        finally:
            import os
            os.unlink(tmp_path)

    with gr.Blocks(title="药品说明书智能识别与语音播报") as demo:
        gr.Markdown("# 药品说明书智能识别与语音播报系统")
        gr.Markdown("上传药品说明书图片，系统将自动识别文字、提取关键信息并语音播报，帮助老年人看清读懂药品说明书。")

        with gr.Row():
            with gr.Column(scale=1):
                image_input = gr.Image(label="药品说明书图片", type="filepath")

                with gr.Accordion("图片分割设置", open=True):
                    enable_split = gr.Checkbox(value=True, label="启用图片分割（文字太小时建议开启）")
                    num_splits = gr.Dropdown(choices=[4, 9, 16], value=4, label="分割数量")
                    overlap_ratio = gr.Slider(minimum=0.0, maximum=0.3, value=0.1, step=0.05, label="重叠比例")

                with gr.Accordion("生成参数设置", open=True):
                    ocr_max_tokens = gr.Slider(minimum=100, maximum=8192, value=ocr_max_new_tokens, step=1, label="OCR 最大生成 token 数")
                    llm_max_tokens = gr.Slider(minimum=100, maximum=4096, value=llm_max_new_tokens, step=1, label="LLM 最大生成 token 数")

                run_btn = gr.Button("开始识别", variant="primary")

            with gr.Column(scale=1):
                ocr_output = gr.Textbox(label="OCR 识别结果", lines=10, max_lines=20)
                info_output = gr.Textbox(label="关键信息整理", lines=15, max_lines=30)
                audio_output = gr.Audio(label="语音播报", type="filepath")

        run_btn.click(
            fn=gradio_pipeline,
            inputs=[
                image_input,
                enable_split,
                num_splits,
                overlap_ratio,
                ocr_max_tokens,
                llm_max_tokens,
            ],
            outputs=[ocr_output, info_output, audio_output],
        )

    return demo

print("✅ 管线编排与 Gradio 界面定义完成 (子进程模式)")

✅ 管线编排与 Gradio 界面定义完成 (子进程模式)


## 主流程
[返回目录 ⬆️](#目录：)

主流程包含以下步骤：
1. 加载图片
2. 图片分割（可选，针对文字太小的说明书，将图片切割成多部分进行识别，分割的图片有重叠）
3. OCR 文字识别（**在子进程中加载模型，完成后销毁子进程**）
4. 大模型文字整理（**在子进程中加载模型，完成后销毁子进程**）
5. 语音合成（**在子进程中加载模型，完成后销毁子进程**）

> 每个步骤在独立的子进程中执行，子进程完成后自动销毁，确保模型内存完全释放。

In [7]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

print("✅ 日志配置完成 (级别: INFO)")

✅ 日志配置完成 (级别: INFO)


In [8]:
# 模型管理器已移除 - 现在使用子进程模式
# 每个模型在独立的子进程中加载、执行、然后自动销毁
print("✅ 子进程模式已启用 - 模型将在需要时自动加载和释放")

✅ 子进程模式已启用 - 模型将在需要时自动加载和释放


In [ ]:
from pathlib import Path

sample_image_path = str(Path("resource/1.jpg"))

result = drug_ocr_pipeline(
    ocr_model_dir=ocr_model_dir,
    llm_model_dir=llm_model_dir,
    image_path=sample_image_path,
    enable_split=False,
    num_splits=4,
    overlap_ratio=0.1,
    ocr_max_new_tokens=ocr_max_new_tokens,
    llm_max_new_tokens=llm_max_new_tokens,
)

print("\n" + "=" * 60)
print("📋 OCR 识别结果:")
print("=" * 60)
print(result["ocr_text"][:500] + "..." if len(result["ocr_text"]) > 500 else result["ocr_text"])

print("\n" + "=" * 60)
print("📝 大模型整理结果:")
print("=" * 60)
print(result["extracted_info"])

# 播放音频
if result["audio"] is not None:
    import IPython.display as ipd
    sr, wav_data = result["audio"]
    print("\n🔊 播放语音...")
    ipd.display(ipd.Audio(wav_data, rate=sr))

14:14:16 [drug_ocr] INFO: ============================================================
14:14:16 [drug_ocr] INFO: 药品说明书识别管线启动 (子进程模式)
14:14:16 [drug_ocr] INFO:   图片路径: resource/1.jpg
14:14:16 [drug_ocr] INFO:   图片分割: False (num_splits=4, overlap=0.10)
14:14:16 [drug_ocr] INFO: ============================================================
14:14:16 [drug_ocr] INFO: [OCR Step] 加载图片...
14:14:16 [drug_ocr] INFO: [OCR Step] 图片加载完成, 尺寸: (2014, 2881)
14:14:16 [drug_ocr] INFO: [OCR Step] 跳过图片分割
14:14:17 [drug_ocr] INFO: [OCR Step] 启动 OCR 子进程...
I0503 14:14:18.096459 1035908 init.cc:238] ENV [CUSTOM_DEVICE_ROOT]=/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device
I0503 14:14:18.096537 1035908 init.cc:146] Try loading custom device libs from: [/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device]
I0503 14:14:18.217633 1035908 custom_device_load.cc:51] Succeed in loading custom runtime in lib: /opt/conda/envs/python35-paddle120

[OCR Worker] 加载 OCR 模型 (PaddleOCR-VL)...


INFO     2026-05-03 14:14:21,132 1035908 args_utils.py[line:639] Parameter `engine_worker_queue_port` is not specified, found available ports for possible use: [28305]
INFO     2026-05-03 14:14:21,134 1035908 args_utils.py[line:639] Parameter `cache_queue_port` is not specified, found available ports for possible use: [38724]
INFO     2026-05-03 14:14:21,136 1035908 args_utils.py[line:639] Parameter `rdma_comm_ports` is not specified, found available ports for possible use: [14751]
INFO     2026-05-03 14:14:21,139 1035908 args_utils.py[line:639] Parameter `pd_comm_port` is not specified, found available ports for possible use: [19484]
INFO     2026-05-03 14:14:21,140 1035908 download.py[line:142] Using download source: huggingface
INFO     2026-05-03 14:14:21,141 1035908 configuration_utils.py[line:1215] Loading configuration file baidu/PaddleOCR-VL-1.5/config.json
WARNING  2026-05-03 14:14:21,143 1035908 configuration_utils.py[line:1246] You are using a model of type paddleocr_vl to i

current sm_version=71


WARNING  2026-05-03 14:14:22,285 1035908 moe.py[line:41] import noaux_tc Failed!
INFO     2026-05-03 14:14:24,261 1035908 download.py[line:142] Using download source: huggingface
INFO     2026-05-03 14:14:24,264 1035908 configuration_utils.py[line:425] Loading configuration file baidu/PaddleOCR-VL-1.5/generation_config.json
INFO     2026-05-03 14:14:24,284 1035908 tokenizer_utils.py[line:257] Using download source: huggingface
INFO     2026-05-03 14:14:25,941 1035908 engine.py[line:151] Waiting for worker processes to be ready...
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Loading Layers: 100%|██████████| 100/100 [00:00<00:00, 198.88it/s]   
INFO     2026-05-03 14:14:39,443 1035908 engine.py[line:209] Worker pr

...

[LLM Worker] LLM 模型加载完成, 耗时: 15.08s
[LLM Worker] 正在生成回复 (max_new_tokens=200)...


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s][2026-05-03 14:15:20] [1076624] [INFO] Prefill batch, dp_rank: 0, #new-seq: 1, #new-token: 1, #cached-token: 0, token usage: 0.00, #running-req: 1, #queue-req: 0, 
[2026-05-03 14:15:22] [1076624] [INFO] Decode batch, dp_rank: 0, #running-req: 1, #token: 176, token usage: 0.00, cuda graph: False, gen throughput (token/s): 8.23, #queue-req: 0, 
Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.45s/it, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


[LLM Worker] 信息提取完成, 生成耗时: 3.46s, 结果长度: 289
>>> 在这样一支粉色的手指往前一拉，我像一只蝴蝶似的飞到了你的身边

你轻轻地将我的手贴在脸颊，柔软的触感瞬间让我一下子陷了进去

“喜欢就好，别舍不得，我们一起去海边好不好？”

我微微一笑，眼神带着一丝甜蜜，嘴角不自觉地扬起了

“好，那就一起去，我保证不弄疼你，我们一起海边，好不好？”

我环住你，紧紧地靠在你的身上，感受着你的温度和怀抱的柔软

你轻轻地将我搂入怀中，仿佛一只受伤的小动物，任由我紧紧地依靠着你

随着一阵海风轻拂，我们来到了海边

风轻轻掀起了我的长发，海浪一波一波地涌来

我仰头看着那片广阔无垠的蓝，心中满是向往

“这就是我想要的，这是我第一次来这里
[LLM Worker] LLM 模型已释放


14:15:30 [drug_ocr] INFO: [LLM Step] LLM 信息提取完成, 结果长度: 289, 耗时: 28.24s
14:15:30 [drug_ocr] INFO: [TTS Step] TTS 语音合成...
14:15:30 [drug_ocr] INFO: [TTS Step] 启动 TTS 子进程...
I0503 14:15:30.627210 1088334 init.cc:238] ENV [CUSTOM_DEVICE_ROOT]=/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device
I0503 14:15:30.627287 1088334 init.cc:146] Try loading custom device libs from: [/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device]
I0503 14:15:30.751516 1088334 custom_device_load.cc:51] Succeed in loading custom runtime in lib: /opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device/libpaddle-iluvatar-gpu.so
I0503 14:15:30.751560 1088334 custom_device_load.cc:58] Skipped lib [/opt/conda/envs/python35-paddle120-env/lib/python3.10/site-packages/paddle_custom_device/libpaddle-iluvatar-gpu.so]: no custom engine Plugin symbol in this lib.
I0503 14:15:30.759230 1088334 custom_kernel.cc:68] Succeed

[TTS Worker] 加载 TTS 模型 (PaddleSpeech)...
[TTS Worker] TTS 模型加载完成, 耗时: 0.00s
[TTS Worker] 语音合成开始, 输入文字长度: 289


## Gradio 交互界面
[返回目录 ⬆️](#目录：)

通过 Gradio 界面，用户可以：
- 上传药品说明书图片
- 设置是否启用图片分割及分割数量
- 调整各模型的生成参数（max_new_tokens）
- 查看识别和整理结果
- 播放语音合成的音频

> 每次点击"开始识别"时，各模型在独立子进程中执行，完成后自动销毁子进程释放内存。

In [ ]:
demo = make_demo(
    ocr_model_dir=ocr_model_dir,
    llm_model_dir=llm_model_dir,
    ocr_max_new_tokens=ocr_max_new_tokens,
    llm_max_new_tokens=llm_max_new_tokens,
)

try:
    demo.launch(server_name="0.0.0.0", server_port=7860, debug=True)
except Exception:
    demo.launch(debug=True, share=True)